In [0]:
# ==============================================================================
# TASK 8.1 STEP 1: CREATE HIGHLY FRAGMENTED (SMALL FILES) DELTA TABLE
# ==============================================================================

# Ensure catalog and schema context
spark.sql("USE CATALOG globalmart")
spark.sql("USE SCHEMA bronze")

# 1. Drop temporary optimization table if it already exists
spark.sql("DROP TABLE IF EXISTS globalmart.bronze.orders_small_files")

# 2. Read bronze orders and repartition into 100 small files
df_orders = spark.table("globalmart.bronze.bronze_orders")

df_orders.repartition(100) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("globalmart.bronze.orders_small_files")

print("✅ Fragmented Delta table 'globalmart.bronze.orders_small_files' created with 100 small files!")

In [0]:
%sql
-- TASK 8.1 STEP 2: DESCRIBE DETAIL BEFORE OPTIMIZATION
DESCRIBE DETAIL globalmart.bronze.orders_small_files;

In [0]:
%sql
-- TASK 8.1 STEP 3: RUN COMPACTION
OPTIMIZE globalmart.bronze.orders_small_files;

In [0]:
%sql
-- TASK 8.1 STEP 4: DESCRIBE DETAIL AFTER OPTIMIZATION
DESCRIBE DETAIL globalmart.bronze.orders_small_files;

In [0]:
# ==============================================================================
# TASK 8.2 STEP 1: CREATE PARTITIONED FACT TABLE
# ==============================================================================

from pyspark.sql import functions as F

# 1. Set Catalog and Schema Context
spark.sql("USE CATALOG globalmart")
spark.sql("USE SCHEMA bronze")

# 2. Read bronze orders and create order_purchase_month column
df_orders = spark.table("globalmart.bronze.bronze_orders") \
    .withColumn("order_purchase_month", F.date_format(F.col("order_purchase_timestamp"), "yyyy-MM"))

# 3. Write partitioned Delta table
df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_purchase_month") \
    .saveAsTable("globalmart.bronze.orders_partitioned")

print("✅ Partitioned table 'globalmart.bronze.orders_partitioned' created successfully!")

In [0]:
%sql
SHOW TABLES IN globalmart.bronze LIKE 'orders_partitioned';

In [0]:
%sql
SHOW TABLES IN globalmart.bronze LIKE 'orders_partitioned';

In [0]:
%sql
-- Verify Partition Pruning
EXPLAIN EXTENDED
SELECT * 
FROM globalmart.bronze.orders_partitioned 
WHERE order_purchase_month = '2018-01';

In [0]:
%sql
-- Verify Partition Pruning
EXPLAIN EXTENDED
SELECT * 
FROM globalmart.bronze.orders_partitioned 
WHERE order_purchase_month = '2018-01';

In [0]:
%sql
-- TASK 8.3 STEP 1: INSPECT COMMIT HISTORY
DESCRIBE HISTORY globalmart.bronze.orders_partitioned;

In [0]:
%sql
-- TASK 8.3 STEP 2: QUERY HISTORICAL VERSION 0
SELECT COUNT(*) AS version_0_record_count 
FROM globalmart.bronze.orders_partitioned VERSION AS OF 0;

In [0]:
%sql
-- TASK 8.3 STEP 3: RUN STANDARD VACUUM CLEANUP
VACUUM globalmart.bronze.orders_partitioned;

In [0]:
%sql
-- Deliverable 1: Query historical version before OPTIMIZE/Z-ORDER
SELECT COUNT(*) AS version_0_record_count 
FROM globalmart.bronze.orders_partitioned VERSION AS OF 0;

In [0]:
%sql
-- Deliverable 2: Query active current version
SELECT COUNT(*) AS current_record_count 
FROM globalmart.bronze.orders_partitioned;

In [0]:
%sql
VACUUM globalmart.bronze.orders_partitioned DRY RUN;

In [0]:
%sql
VACUUM globalmart.bronze.orders_partitioned DRY RUN;

In [0]:
%sql
DESCRIBE HISTORY globalmart.bronze.orders_partitioned;

In [0]:
%sql
-- 1. Baseline Aggregate Count
SELECT COUNT(*) AS original_count 
FROM globalmart.bronze.orders_partitioned;

In [0]:
%sql
-- 2. Identify Baseline Version Number
SELECT max(version) AS baseline_version 
FROM (DESCRIBE HISTORY globalmart.bronze.orders_partitioned);

In [0]:
%sql
-- Delete a subset of data to modify table state
DELETE FROM globalmart.bronze.orders_partitioned 
WHERE order_purchase_month = '2018-01';

In [0]:
%sql
-- Aggregate count after modification
SELECT COUNT(*) AS count_after_modification 
FROM globalmart.bronze.orders_partitioned;

In [0]:
%sql
-- Query baseline state via Time Travel (Replace 4 with your baseline_version if different)
SELECT COUNT(*) AS count_from_old_version 
FROM globalmart.bronze.orders_partitioned VERSION AS OF 4;

In [0]:
%sql
-- Restore table to baseline version
RESTORE TABLE globalmart.bronze.orders_partitioned TO VERSION AS OF 4;

In [0]:
%sql
-- Verify aggregate count after restore
SELECT COUNT(*) AS count_after_restore 
FROM globalmart.bronze.orders_partitioned;

In [0]:
# ==============================================================================
# TASK 8.5 STEP 1: CREATE LIQUID CLUSTERED TABLE
# ==============================================================================

# 1. Set context
spark.sql("USE CATALOG globalmart")
spark.sql("USE SCHEMA bronze")

# 2. Drop existing table if present
spark.sql("DROP TABLE IF EXISTS globalmart.bronze.orders_liquid")

# 3. Read bronze orders base table
df_orders = spark.table("globalmart.bronze.bronze_orders")

# 4. Write table with Liquid Clustering enabled
df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("clusterBy", "order_purchase_timestamp, customer_id") \
    .saveAsTable("globalmart.bronze.orders_liquid")

print("✅ Liquid Clustered table 'globalmart.bronze.orders_liquid' created successfully!")

In [0]:
%sql
-- TASK 8.5 STEP 2: RUN INITIAL LIQUID OPTIMIZE
OPTIMIZE globalmart.bronze.orders_liquid;

In [0]:
%sql
-- TASK 8.5 STEP 3: DESCRIBE DETAIL FOR METADATA
DESCRIBE DETAIL globalmart.bronze.orders_liquid;

In [0]:
# ==============================================================================
# TASK 8.5 STEP 4: APPEND NEW DATA TO SIMULATE DATA GROWTH
# ==============================================================================

# Read a subset of orders and append to simulate incremental write
df_append = spark.table("globalmart.bronze.bronze_orders").limit(10000)

df_append.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("globalmart.bronze.orders_liquid")

print("✅ Appended 10,000 new records to orders_liquid!")

In [0]:
%sql
-- TASK 8.5 STEP 4 (Cont): INCREMENTAL OPTIMIZE
OPTIMIZE globalmart.bronze.orders_liquid;

In [0]:
%sql
-- Compare Execution Plan for Liquid Clustered Table
EXPLAIN EXTENDED
SELECT * 
FROM globalmart.bronze.orders_liquid 
WHERE customer_id = '9ef432eb6251297304e76186ac53a3fe'
  AND order_purchase_timestamp >= '2018-01-01';